# ERIA Phase 4 — Topic Classification (Zero-Shot)

## Objective
This notebook implements **Layer 2: Hugging Face Topic Classifier** for the ERIA pipeline.

We use a zero-shot classification approach with:
- Model: `facebook/bart-large-mnli`
- Task: Assign semantic labels to regulatory text chunks

---

## Goals
- Classify each chunk into predefined regulatory categories
- Analyze label distribution
- Validate classification quality
- Prepare inputs for downstream LLM reasoning (Phase 5)

---

## Input
- Preprocessed chunks (Phase 3 output)

## Output
- Classified chunks with:
  - label
  - confidence score

In [ ]:
import os
import sys
import pandas as pd
from collections import Counter

# Add project root
sys.path.append(os.path.abspath(".."))

from src.analysis.classifier import TopicClassifier
from src.utils.config import BASE_DIR
from src.utils.helpers import load_json
from src.utils.logger import get_logger

logger = get_logger("LLMExperimentsNotebook")
logger.info("Notebook started")

## Load Preprocessed Chunks

We load chunked data generated from Phase 3 preprocessing.

In [ ]:
input_path = os.path.join(
    BASE_DIR,
    "artifacts",
    "chunks",
    "ERIA_Sample_1.json"
)

data = load_json(input_path)

print(f"Total chunks loaded: {len(data)}")
print("\nSample chunk:\n")
print(data[0]["text"][:300])

## Initialize Zero-Shot Classifier

We use Hugging Face's `facebook/bart-large-mnli` model for zero-shot classification.

In [ ]:
classifier = TopicClassifier()

## Run Classification

Each chunk is classified into one of the predefined categories.

In [ ]:
results = classifier.process_chunks(data)

print("Sample Output:\n")
results[:3]

## Convert Results to DataFrame

This helps in analysis and visualization.

In [ ]:
df = pd.DataFrame(results)

df.head()

## Label Distribution

We analyze how chunks are distributed across categories.

In [ ]:
label_counts = Counter(df["label"])

label_counts

In [ ]:
pd.Series(label_counts).sort_values(ascending=False)

## Confidence Score Analysis

Understanding how confident the model is in its predictions.

In [ ]:
df["confidence"].describe()

In [ ]:
df["confidence"].hist(bins=20)

In [ ]:
low_conf = df[df["confidence"] < 0.6]

print(f"Low confidence samples: {len(low_conf)}")
low_conf.head()

## Inspect Samples by Label

Manual inspection to validate classification quality.

In [ ]:
label = "Financial Benefits"

samples = df[df["label"] == label].head(3)

for i, row in samples.iterrows():
    print(f"\n--- Chunk {row['chunk_id']} ---")
    print("Confidence:", row["confidence"])
    print(row["text"][:500])

## Save Classified Output

This output will be used in Phase 5 (LLM-based analysis).

In [ ]:
output_path = os.path.join(
    BASE_DIR,
    "artifacts",
    "llm_outputs",
    "classified_chunks.json"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

import json

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Saved to:", output_path)

## Key Observations

- Majority of chunks fall under:
  - Implementation Process
  - Eligibility Criteria
  - Administrative Structure

- Financial Benefits and Application Process are clearly identifiable.

- Some chunks may have:
  - Mixed context
  - Lower confidence due to OCR noise

---

## Limitations

- Zero-shot classification depends heavily on text quality
- OCR noise can reduce accuracy
- Some chunks contain overlapping topics

---

## Next Step (Phase 5)

We will:
- Group chunks by label
- Feed structured context to LLM (Gemini / Groq)
- Generate:
  - summaries
  - impact analysis
  - stakeholder insights

#  ERIA — LLM Experiments (Phase 5)

## Objective
Evaluate the performance of LLM-based analysis on processed education regulation documents.

## Goals
- Generate structured outputs using Gemini API
- Validate JSON consistency
- Assess response quality for:
  - Summary
  - Stakeholder impact
  - Risk detection
  - Timeline extraction

## Input
- Preprocessed + chunked regulation data

## Output
- Structured JSON from LLM

In [1]:
import os
import sys
import json

sys.path.append(os.path.abspath(".."))

from src.utils.config import BASE_DIR
from src.utils.helpers import load_json
from src.utils.logger import get_logger

from src.llm.client import GroqClient
from src.llm.prompts import build_prompt
from src.llm.parser import LLMParser

logger = get_logger("LLMExperiments")
logger.info("Notebook started")

2026-04-27 23:35:14,983 - LLMExperiments - INFO - Notebook started


In [3]:
input_path = os.path.join(
    BASE_DIR,
    "artifacts",
    "chunks",
    "ERIA_Sample_1.json"
)

data = load_json(input_path)

print(f"Total chunks loaded: {len(data)}")
print("\nSample chunk:\n")
print(data[0]["text"][:300])

Total chunks loaded: 55

Sample chunk:

ogiat -Pears fagaaa frataenaa ayer ART University Grants Commission area : (TSreN FATA, AKA AHI) Prof. .6-1/2026(interniship) 26 7, 1948 / 16 April, 2026 Subject: Information regarding Guidelines the Prime Minister Internship Scheme (PMIS) aretha agiqa / Aetea, The Ministry Corporate Affairs has lau


##  Combine Chunks for Single LLM Call

To reduce latency and cost, we merge chunks into a single input.

In [4]:
full_text = " ".join([c["text"] for c in data])

print("Total text length:", len(full_text))

Total text length: 40135


##  Input Truncation Strategy

Due to LLM token limits, we restrict input size.

In [7]:
# Step 1: Define important keywords
important_keywords = [
    "eligibility", "criteria", "scheme", "implementation",
    "impact", "guidelines", "regulation", "requirements",
    "internship", "benefits", "conditions", "rules"
]

# Step 2: Filter relevant lines
filtered_lines = [
    line for line in full_text.split("\n")
    if any(k in line.lower() for k in important_keywords)
]

# Step 3: Join back into text
filtered_text = "\n".join(filtered_lines)

In [8]:
MAX_CHARS = 12000
input_text = filtered_text[:MAX_CHARS]

print("Filtered input length:", len(input_text))

Filtered input length: 12000


##  LLM Inference (Gemini)

We generate structured analysis using a single LLM call.

In [9]:
client = GroqClient()

prompt = build_prompt(input_text)

response = client.generate(prompt)

print(response[:1000])

2026-04-27 23:35:32,730 - src.llm.client - ERROR - Rate limit hit. Skipping retries.


##  Parsing LLM Output

Convert raw LLM response into structured JSON.

In [7]:
print(response)

{{ 
  "regulation_topic": "Prime Minister Internship Scheme",
  "chronology": {
    "predecessor_circulars": ["The Prime Minister Internship Scheme was announced in the Budget 2024-25"],
    "amendments": ["The Pilot Project has now been extended till December 2026"],
    "historical_context": ["The scheme aims to provide internship opportunities to youth in top companies"]
  },
  "impact_analysis": {
    "students": ["Eligible final-year graduate and postgraduate students can participate in the scheme"],
    "faculty": ["Faculty members may be involved in guiding students to participate in the scheme"],
    "institutions": ["Institutions can promote the scheme to their students and facilitate participation"],
    "administrators": ["Administrators will be responsible for implementing and monitoring the scheme"],
    "accreditation_compliance": ["The scheme is distinct from existing schemes related to skill development and apprenticeships"]
  },
  "sentiment_risk": {
    "positive_indi

In [8]:
response = response.replace("{{", "{").replace("}}", "}")

In [9]:
parsed = LLMParser.parse(response)

print(parsed.keys())

dict_keys(['regulation_topic', 'chronology', 'impact_analysis', 'sentiment_risk', 'summary', 'stakeholder_report', 'impact_assessment', 'positives', 'negatives'])


## 💾 Save LLM Output

Persist structured results for downstream analysis.

In [10]:
output_path = os.path.join(
    BASE_DIR,
    "artifacts",
    "llm_outputs",
    "eria_analysis.json"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(parsed, f, indent=2)

print("Saved to:", output_path)

Saved to: F:\DATA SCIENCE\Projects\Education Regulation Impact Analyzer (ERIA)\artifacts\llm_outputs\eria_analysis.json


## 📊 Observations

- JSON structure consistency: ✅ / ❌
- Summary clarity: ⭐⭐⭐⭐☆
- Stakeholder coverage: ⭐⭐⭐⭐☆
- Risk detection quality: ⭐⭐⭐☆☆

## Issues Noticed
- Possible hallucination in timeline (if any)
- Missing stakeholders in some cases

## Next Steps
- Introduce chunk filtering (Phase 5.5)
- Improve prompt constraints